In [2]:
# Install the missing high-performance fuzzy matching library
!pip install rapidfuzz

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 1.0 MB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.5 MB 1.3 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.4 MB/s  0:00:01


In [4]:
#Project Initialization & Security Setup
import sqlite3
import pandas as pd
from rapidfuzz import process, utils
import urllib.parse
from getpass import getpass

 # This prompts you to enter it privately when the cell runs.
GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

Enter your Gemini API Key:  ········


In [5]:
# Verify the key exists (without printing the whole secret)
if 'GEMINI_API_KEY' in globals() and len(GEMINI_API_KEY) > 10:
    print(f"✅ Success! Key is stored. Length: {len(GEMINI_API_KEY)} characters.")
else:
    print("❌ Key not found. Try running the getpass cell again.")

✅ Success! Key is stored. Length: 39 characters.


In [6]:
#Secure Database & "Synthetic" Data Generation
def setup_database():
    conn = sqlite3.connect('bazaar_bridge.db')
    cursor = conn.cursor()
    
    # Create a secure Inventory table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS inventory (
            id INTEGER PRIMARY KEY,
            product_name TEXT,
            base_price REAL,
            stock_qty INTEGER
        )
    ''')
    
    # Synthetic Data: High-volume Jodia Bazaar items
    # Using a "Markup Factor" logic later for security
    sample_data = [
        (1, 'Lal Mirch (Red Chili)', 1200.0, 50),
        (2, 'Haldi (Turmeric)', 800.0, 100),
        (3, 'Sufaied Zeera (Cumin)', 1500.0, 30),
        (4, 'Kali Mirch (Black Pepper)', 2200.0, 20),
        (5, 'Dhaniya Powder (Coriander)', 600.0, 80)
    ]
    
    # Secure bulk insert
    cursor.executemany('INSERT OR REPLACE INTO inventory VALUES (?,?,?,?)', sample_data)
    conn.commit()
    return conn

db_conn = setup_database()
print("✔ Database initialized securely.")

✔ Database initialized securely.


In [7]:
def get_product_list(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT product_name FROM inventory")
    return [row[0] for row in cursor.fetchall()]

def process_order_item(user_input, conn):
    products = get_product_list(conn)
    
    # Process "Messy" input (e.g., "Lal Mirhc" -> "Lal Mirch")
    match = process.extractOne(user_input, products, score_cutoff=65)
    
    if match:
        matched_name = match[0]
        # Securely fetch price using parameterized query
        cursor = conn.cursor()
        cursor.execute("SELECT base_price FROM inventory WHERE product_name = ?", (matched_name,))
        base_price = cursor.fetchone()[0]
        
        # Portfolio Security: Apply 10% Markup instead of showing real wholesale rates
        retail_price = round(base_price * 1.10, 2)
        return {"item": matched_name, "price": retail_price, "status": "Success"}
    
    return {"status": "Not Found"}

# Test it:
test_input = "Lal Mirhc" # Intentional typo
result = process_order_item(test_input, db_conn)
print(f"Input: {test_input} | Identified as: {result['item']} | Portfolio Price: {result['price']}")

Input: Lal Mirhc | Identified as: Lal Mirch (Red Chili) | Portfolio Price: 1320.0


In [ ]:
#"This output confirms my 'Headless' engine works. It takes a misspelled Roman Urdu input, cleans it using a local fuzzy-matching algorithm to save API costs, securely retrieves the price from an SQLite database using parameterized queries, and applies a dynamic markup before presenting the final quote."

In [8]:
#The Agentic Bridge: WhatsApp Simulation
def generate_order_link(customer_name, order_items):
    phone = "923001234567" # Placeholder
    
    # Build a professional Roman Urdu / English message
    msg_body = f"Assalam-o-Alaikum! Bazaar-Bridge Order for {customer_name}:\n"
    total = 0
    for item in order_items:
        msg_body += f"- {item['name']}: Rs.{item['price']}\n"
        total += item['price']
    
    msg_body += f"\nTotal (incl. tax): Rs.{total}\nPlease confirm."
    
    # Encode for URL security
    encoded_msg = urllib.parse.quote(msg_body)
    return f"https://wa.me/{phone}?text={encoded_msg}"

# Example Usage
order_list = [{"name": "Lal Mirch", "price": 1320.0}]
link = generate_order_link("Ali Khan", order_list)
print(f"Click to open WhatsApp Simulation:\n{link}")

Click to open WhatsApp Simulation:
https://wa.me/923001234567?text=Assalam-o-Alaikum%21%20Bazaar-Bridge%20Order%20for%20Ali%20Khan%3A%0A-%20Lal%20Mirch%3A%20Rs.1320.0%0A%0ATotal%20%28incl.%20tax%29%3A%20Rs.1320.0%0APlease%20confirm.


In [15]:
import google.generativeai as genai

# 1. Re-configure
genai.configure(api_key=GEMINI_API_KEY)

# 2. Automatically find a valid model name
available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

if available_models:
    # We prefer Flash if available, otherwise take the first one
    selected_model_name = next((m for m in available_models if "flash" in m), available_models[0])
    model = genai.GenerativeModel(selected_model_name)
    print(f"✅ Connection Successful! Using model: {selected_model_name}")
else:
    print("❌ No generative models found for this API key. Please check your Google AI Studio billing/plan.")

# 3. Quick test
try:
    print("🤖 AI Test Response:", model.generate_content("Hi").text)
except Exception as e:
    print(f"⚠️ Connection check failed: {e}")

✅ Connection Successful! Using model: models/gemini-2.5-flash
🤖 AI Test Response: Hi there! How can I help you today?


In [16]:
def run_bazaar_bridge():
    print("--- 🛒 Bazaar-Bridge Terminal ---")
    customer = input("Enter Customer Name: ")
    item_query = input("What are they looking for? (e.g., 'Lal Mirhc'): ")
    
    # 1. Process Logic
    result = process_order_item(item_query, db_conn)
    
    if result["status"] == "Success":
        # 2. Use Gemini Brain to draft a polite Roman Urdu greeting
        prompt = f"Write a 1-line friendly shopkeeper greeting in Roman Urdu for {customer} buying {result['item']}."
        ai_response = model.generate_content(prompt)
        
        print(f"\n✅ Match Found: {result['item']}")
        print(f"🤖 AI Suggestion: {ai_response.text}")
        
        # 3. Generate Link
        final_link = generate_order_link(customer, [{"name": result['item'], "price": result['price']}])
        print(f"\n🔗 Final WhatsApp Link: {final_link}")
    else:
        print("❌ Product not found. Try 'Haldi' or 'Zeera'.")

# Run the full system
run_bazaar_bridge()

--- 🛒 Bazaar-Bridge Terminal ---


Enter Customer Name:  Zeeshan
What are they looking for? (e.g., 'Lal Mirhc'):  Haldi



✅ Match Found: Haldi (Turmeric)
🤖 AI Suggestion: **"Aiye Zeeshan bhai, Haldi chahiye kya?"**

*(Translation: "Come Zeeshan brother, do you need turmeric?")*

🔗 Final WhatsApp Link: https://wa.me/923001234567?text=Assalam-o-Alaikum%21%20Bazaar-Bridge%20Order%20for%20Zeeshan%3A%0A-%20Haldi%20%28Turmeric%29%3A%20Rs.880.0%0A%0ATotal%20%28incl.%20tax%29%3A%20Rs.880.0%0APlease%20confirm.


In [17]:
# Final Step: Close the connection gracefully
try:
    db_conn.close()
    print("✔ Database connection closed safely. Project Complete!")
except:
    pass
    

✔ Database connection closed safely. Project Complete!


In [ ]:
#Executive Summary
#This project successfully demonstrates a Headless AI architecture designed to digitize the traditional wholesale workflows of Jodia Bazaar, Karachi. By combining local high-performance logic with cloud-based LLM reasoning, the system processes "Messy" Roman Urdu inputs into structured business transactions with minimal hardware overhead.

#Key Technical Achievements
#Agentic Scaffolding: Integrated Gemini 1.5 Flash to act as a "Cultural Translator," converting raw data into localized, persona-driven Roman Urdu greetings (e.g., "Aiye Zeeshan bhai...").

#Fuzzy Logic Layer: Leveraged RapidFuzz to handle real-world phonetic misspellings (e.g., "Haldii" → "Haldi"), reducing the need for expensive API calls for basic text cleaning.

#Secure Data Management: Implemented an SQLite backend using Parameterized Queries to ensure 100% protection against SQL Injection attacks while maintaining a low RAM footprint on a Dell Latitude 5490.

#Data Privacy & Ethics: Developed a Markup Logic layer to calculate retail prices dynamically, demonstrating the ability to handle sensitive wholesale "Base Rates" securely in a portfolio environment.

#Zero-Cost Frontend: Utilized URI Encoding to generate universal WhatsApp links, creating a frictionless transition from Python logic to customer communication without requiring a heavy GUI.